# On-Device AI 실전 시험 대비 연습 문제

신규 AVD 교육 커리큘럼(Quantization, Advanced Quantization, Knowledge Distillation, KV Cache Pruning, Speculative Decoding) 기반 연습 문제입니다.
각 문제의 `[문제]`와 `[요구사항]`을 읽고 코드의 `### 작성 필요 ###` 부분을 채워 넣으세요.

In [ ]:
# [코드]
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


## 문제 1. (a-1) Linear Quantization (FP → INT)
[문제]
Q a-1) 부동소수점 텐서(fp_tensor)를 선형 양자화(Linear Quantization)하여 지정된 비트 폭(bitwidth)의 정수 텐서로 변환합니다.

[요구사항]
- fp_tensor를 scale로 나눈 후 torch.round()를 이용해 반올림 수행
- 지정된 dtype 변환 후 zero_point를 더해 영점 이동(shift)
- 양자화 비트 폭에 맞는 정수 범위 [quantized_min, quantized_max]로 clamp 처리
- ※ 사전 정의 함수: get_quantized_range(bitwidth) -> (quantized_min, quantized_max) 가 제공됩니다.

In [ ]:
# [코드]
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8) -> torch.Tensor:
    assert fp_tensor.dtype == torch.float
    scaled_tensor = ### 작성 필요 ###
    rounded_tensor = ### 작성 필요 ###
    rounded_tensor = rounded_tensor.to(dtype)
    shifted_tensor = ### 작성 필요 ###
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    return quantized_tensor

## 문제 2. (a-2) Quantization Scale & Zero Point 계산
[문제]
Q a-2) 입력 텐서의 최댓값/최솟값과 비트 폭(bitwidth)을 기반으로 양자화 스케일(scale)과 제로 포인트(zero_point)를 산출합니다.

[요구사항]
- 실수 범위(fp_max - fp_min)와 정수 범위(quantized_max - quantized_min)의 비율로 scale 계산
- 실수의 최솟값(fp_min)이 정수의 최솟값(quantized_min)에 대응되도록 zero_point 산출 후 반올림
- zero_point가 정수 유효 범위 [quantized_min, quantized_max]를 벗어나지 않도록 클리핑 처리
- ※ 사전 정의 함수: get_quantized_range(bitwidth) -> (quantized_min, quantized_max) 가 제공됩니다.

In [ ]:
# [코드]
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    scale = ### 작성 필요 ###
    zero_point = ### 작성 필요 ###

    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else:
        zero_point = round(zero_point)
    return scale, int(zero_point)

## 문제 3. (a-3) Per-Channel Weight Quantization
[문제]
Q a-3) 가중치 텐서의 각 출력 채널(dim=0)별로 최적의 스케일을 계산하여 Per-Channel 대칭 양자화를 수행합니다.

[요구사항]
- 출력 채널 축(dim=0)을 순회하며 채널별 서브텐서 추출(tensor.select)
- 서브텐서별 스케일을 산출하여 scale 텐서에 저장하고 브로드캐스팅을 위한 view 형상 맞춤
- ※ 사전 정의 함수: get_quantization_scale_for_weight(weight, bitwidth), linear_quantize(...) 가 제공됩니다.

In [ ]:
# [코드]
def linear_quantize_weight_per_channel(tensor, bitwidth):
    dim_output_channels = 0
    num_output_channels = tensor.shape[dim_output_channels]
    scale = torch.zeros(num_output_channels, device=tensor.device)

    for oc in range(num_output_channels):
        _subtensor = ### 작성 필요 ###
        _scale = get_quantization_scale_for_weight(_subtensor, bitwidth)
        scale[oc] = _scale

    scale_shape = [1] * tensor.dim()
    scale_shape[dim_output_channels] = -1
    scale = ### 작성 필요 ###
    quantized_tensor = linear_quantize(tensor, bitwidth, scale, zero_point=0)
    return quantized_tensor, scale, 0

## 문제 4. (a-4) Quantized Linear: Output 재스케일 & Zero Point 보정
[문제]
Q a-4) 정수 행렬 곱셈 결과(INT32)에 스케일 비율을 곱하고 출력 제로 포인트를 보정하여 INT8 출력 텐서를 완성합니다.

[요구사항]
- weight_scale 텐서를 출력 형상 (1, -1)로 정렬
- 누산 출력에 (input_scale * weight_scale / output_scale) 비율 적용 후 output_zero_point 가산
- feature_bitwidth 범위로 clamp 및 반올림하여 torch.int8로 변환
- ※ 사전 정의 함수: get_quantized_range(feature_bitwidth) -> (qmin, qmax) 가 제공됩니다.

In [ ]:
# [코드]
def quantized_linear(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale):
    output = torch.nn.functional.linear(input.to(torch.int32), weight.to(torch.int32), bias)
    weight_scale_reshaped = ### 작성 필요 ###
    output = ### 작성 필요 ###
    qmin, qmax = get_quantized_range(feature_bitwidth)
    output = output.round().clamp(qmin, qmax).to(torch.int8)
    return output

## 문제 5. (a-5) Quantized Conv2d: Output 재스케일 & Zero Point 보정
[문제]
Q a-5) 2D 정수 합성곱 누적 결과에 bias, 스케일 비율, 출력 제로 포인트를 보정하여 INT8 출력 피처맵을 완성합니다.

[요구사항]
- 4D 피처맵 (N, C, H, W)의 채널 축에 맞게 bias와 weight_scale을 (1, -1, 1, 1)로 reshape
- 스케일 보정 및 output_zero_point 적용 후 feature_bitwidth 범위로 clamp 및 torch.int8 변환
- ※ 사전 정의 함수: get_quantized_range(feature_bitwidth) -> (qmin, qmax) 가 제공됩니다.

In [ ]:
# [코드]
def quantized_conv2d(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale,
                     stride, padding, dilation, groups):
    input = torch.nn.functional.pad(input, padding, 'constant', input_zero_point)
    output = torch.nn.functional.conv2d(input.to(torch.int32), weight.to(torch.int32), None, stride, 0, dilation, groups)
    if bias is not None:
        output = output + ### 작성 필요 ###
    output = ### 작성 필요 ###
    qmin, qmax = get_quantized_range(feature_bitwidth)
    output = output.round().clamp(qmin, qmax).to(torch.int8)
    return output

## 문제 6. (a-6) K-means Quantization & Codebook 디코딩
[문제]
Q a-6) bitwidth에 따른 클러스터 개수를 설정하고 K-means 코드북(centroids, labels)으로부터 양자화된 가중치 텐서를 복원합니다.

[요구사항]
- bitwidth에 해당하는 총 클러스터 개수(2의 bitwidth 제곱) 설정
- 코드북의 centroids와 labels를 인덱싱하여 양자화 텐서 디코딩
- ※ 사전 정의 클래스: Codebook(centroids, labels), KMeans 가 제공됩니다.

In [ ]:
# [코드]
def k_means_quantize(fp32_tensor: torch.Tensor, bitwidth=4, codebook=None):
    if codebook is None:
        n_clusters = ### 작성 필요 ###
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)
    quantized_tensor = ### 작성 필요 ###
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))
    return codebook

## 문제 7. (a-7) K-means Codebook 업데이트 (QAT)
[문제]
Q a-7) 학습을 통해 갱신된 가중치 텐서로부터 각 클러스터 k에 속한 원소들의 평균으로 코드북 중심점(centroids)을 업데이트합니다.

[요구사항]
- 1차원으로 평탄화된 fp32_tensor에서 현재 클러스터 인덱스 k에 해당하는 원소들만 마스킹 추출
- 해당 원소들의 평균값(torch.mean)을 codebook.centroids[k]에 할당
- ※ 사전 정의 클래스: Codebook(centroids, labels) 이 제공됩니다.

In [ ]:
# [코드]
def update_codebook(fp32_tensor: torch.Tensor, codebook: Codebook):
    n_clusters = codebook.centroids.numel()
    fp32_tensor = fp32_tensor.view(-1)
    for k in range(n_clusters):
        codebook.centroids[k] = ### 작성 필요 ###

## 문제 8. (b-1) AWQ: Salient Channel Scaling (상위 1% 중요 채널 스케일링)
[문제]
Q b-1) 활성화 중요도(importance)를 기준으로 상위 1% 채널 인덱스를 찾고, 양자화 전후로 scale_factor 스케일링 및 복원을 수행합니다.

[요구사항]
- torch.topk를 활용하여 importance 텐서에서 상위 1%에 해당하는 채널 인덱스(outlier_mask) 추출
- pseudo_quantize_tensor 수행 후 확대했던 outlier 채널의 가중치를 scale_factor로 나누어 원복
- ※ 사전 정의 함수: pseudo_quantize_tensor(w, n_bit, q_group_size) 가 제공됩니다.

In [ ]:
# [코드]
@torch.no_grad()
def pseudo_quantize_model_weight_scaleup(model, w_bit, q_group_size, input_feat, scale_factor):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            importance = sum(input_feat[n]).float()
            num_samples = int(len(importance) * 0.01)
            outlier_mask = ### 작성 필요 ###
            assert outlier_mask.dim() == 1
            m.weight.data[:, outlier_mask] *= scale_factor
            m.weight.data = pseudo_quantize_tensor(m.weight.data, n_bit=w_bit, q_group_size=q_group_size)
            ### 작성 필요 ###

## 문제 9. (b-2) SmoothQuant: LayerNorm ↔ Linear 난이도 전이
[문제]
Q b-2) LayerNorm의 파라미터를 scale로 축소하고 연결된 Linear 레이어들의 가중치에 동일 scale을 곱해 양자화 난이도를 전이합니다.

[요구사항]
- LayerNorm의 weight와 bias에 대해 인플레이스 나눗셈 연산(div_) 수행
- 연결된 모든 fc 레이어의 weight에 대해 인플레이스 곱셈 연산(mul_) 수행

In [ ]:
# [코드]
@torch.no_grad()
def smooth_ln_fcs_by_scale(ln, fcs, scale):
    if not isinstance(fcs, list):
        fcs = [fcs]
    assert isinstance(ln, nn.LayerNorm)
    for fc in fcs:
        assert isinstance(fc, nn.Linear)

    ### 작성 필요 ###
    ### 작성 필요 ###

    for fc in fcs:
        ### 작성 필요 ###

## 문제 10. (b-3) SmoothQuant: Alpha 기반 Scale Factor 계산
[문제]
Q b-3) 활성화 스케일, 가중치 스케일, 균형 하이퍼파라미터 alpha를 이용해 SmoothQuant의 채널별 스케일 팩터를 계산합니다.

[요구사항]
- pow() 연산을 사용하여 act_scales의 alpha 제곱과 weight_scales의 (1 - alpha) 제곱의 비율 산출
- 산출된 scales의 최솟값 클램핑(1e-5) 후 LayerNorm 및 FC 레이어 파라미터 스케일 업데이트

In [ ]:
# [코드]
@torch.no_grad()
def smooth_ln_fcs(ln, fcs, act_scales, alpha=0.5):
    if not isinstance(fcs, list):
        fcs = [fcs]
    device, dtype = fcs[0].weight.device, fcs[0].weight.dtype
    act_scales = act_scales.to(device=device, dtype=dtype)
    weight_scales = torch.cat(
        [fc.weight.abs().max(dim=0, keepdim=True)[0] for fc in fcs], dim=0
    )
    weight_scales = weight_scales.max(dim=0)[0].clamp(min=1e-5)

    scales = ### 작성 필요 ###
    scales = scales.clamp(min=1e-5).to(device).to(dtype)

    ln.weight.div_(scales)
    ln.bias.div_(scales)
    for fc in fcs:
        fc.weight.mul_(scales.view(1, -1))

## 문제 11. (b-4) Rotation-based Quantization: 직교 행렬 가중치 회전
[문제]
Q b-4) 트랜스포머 모듈(Embedding 및 Linear)의 입출력 방향에 맞게 직교 회전 행렬 R1을 가중치 행렬에 곱하여 회전시킵니다.

[요구사항]
- Embedding 가중치 (num_tokens, hidden_dim)의 출력 차원 축에 R1 행렬 곱 적용
- 출력 프로젝션(o_proj, down_proj) Linear 가중치 (out, in)의 출력 축에 R1의 전치 행렬(R1.T)을 좌측 곱셈
- 그 외 입력 프로젝션(Q, K, V, gate/up) Linear 가중치의 입력 축에 R1을 우측 곱셈

In [ ]:
# [코드]
def rotate_model_weight(model, R1):
    for n, m in model.named_modules():
        if isinstance(m, nn.Embedding):
            W_ = m.weight.data
            m.weight.data = ### 작성 필요 ###
        if isinstance(m, nn.Linear):
            if "o_proj" in n or "down_proj" in n:
                W_ = m.weight.data
                m.weight.data = ### 작성 필요 ###
            else:
                W_ = m.weight.data
                m.weight.data = ### 작성 필요 ###

## 문제 12. (c-1) Knowledge Distillation (Soft Targets KL Loss)
[문제]
Q c-1) Teacher와 Student 모델의 출력 로짓에 온도 T를 적용해 Soft Targets 손실을 계산하고, 실제 정답(Hard Label) 손실과 결합합니다.

[요구사항]
- Teacher 추론 시 기울기 계산 비활성화(torch.no_grad) 컨텍스트 적용
- 온도 T로 나눈 로짓에 대해 마지막 차원(dim=-1) 소프트맥스 확률 분포 생성
- KL 손실 계산 후 T의 제곱을 곱하여 스케일 보정하고, 정답 손실(ce_loss)과 가중합 계산

In [ ]:
# [코드]
def kd_step(teacher, student, inputs, labels, T, soft_target_loss_weight, ce_loss_weight, ce_loss):
    with ### 작성 필요 ###:
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)

    soft_targets = ### 작성 필요 ###
    student_prob = ### 작성 필요 ###

    soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - student_prob.log()), dim=1).mean() * ### 작성 필요 ###
    label_loss = ce_loss(student_logits, labels)
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
    return loss

## 문제 13. (c-2) Cosine Similarity Loss 기반 Knowledge Distillation
[문제]
Q c-2) Teacher와 Student의 중간 은닉 표현 벡터 간 코사인 임베딩 손실을 계산하고 분류 손실과 결합합니다.

[요구사항]
- Teacher 모델로부터 은닉 표현(teacher_hidden_representation)을 기울기 없이 추출
- cosine_loss 호출 시 target 인자로 배치 크기만큼의 1로 채워진 텐서(torch.ones) 전달
- ※ 사전 정의 객체: ce_loss (CrossEntropyLoss), cosine_loss (CosineEmbeddingLoss) 가 주어집니다.

In [ ]:
# [코드]
def cosine_kd_step(teacher, student, inputs, labels, hidden_rep_loss_weight, ce_loss_weight, ce_loss, cosine_loss):
    with torch.no_grad():
        _, teacher_hidden_representation = teacher(inputs)
    student_logits, student_hidden_representation = student(inputs)

    hidden_rep_loss = ### 작성 필요 ###
    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 문제 14. (c-3) Intermediate Regressor 기반 MSE Feature Map Distillation
[문제]
Q c-3) Teacher와 Student(Regressor 통과 출력)의 중간 피처맵 간 평균 제곱 오차(MSE) 손실을 계산합니다.

[요구사항]
- Student가 반환한 regressor_feature_map과 Teacher의 teacher_feature_map 간 MSE 손실 계산
- feature_map_weight와 ce_loss_weight 가중치를 적용하여 총 손실 반환
- ※ 사전 정의 객체: ce_loss (CrossEntropyLoss), mse_loss (MSELoss) 가 주어집니다.

In [ ]:
# [코드]
def mse_kd_step(teacher, student, inputs, labels, feature_map_weight, ce_loss_weight, ce_loss, mse_loss):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)
    student_logits, regressor_feature_map = student(inputs)

    hidden_rep_loss = ### 작성 필요 ###
    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 문제 15. (d-1) KV Cache 메모리 사용량(Bytes) 계산
[문제]
Q d-1) 모델 설정(config)과 시퀀스 길이(seq_len), 배치 크기, 데이터 타입 바이트 수를 바탕으로 KV Cache가 점유하는 총 바이트 수를 계산합니다.

[요구사항]
- Key와 Value 두 종류의 캐시 텐서(계수 2) 모두 포함
- 배치 크기, 레이어 수, KV 헤드 수(GQA 반영), 시퀀스 길이, 헤드 차원, 원소당 바이트 수를 곱하여 총 바이트 산출

In [ ]:
# [코드]
def kv_cache_bytes(config, seq_len, batch_size=1, dtype_bytes=2):
    n_layers = config.num_hidden_layers
    n_kv_heads = config.num_key_value_heads
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)

    total_bytes = ### 작성 필요 ###
    return total_bytes

## 문제 16. (d-2) H2O (Heavy-Hitter Oracle) Attention 누적 스코어링
[문제]
Q d-2) 어텐션 행렬(attn_kv)로부터 키 토큰별 누적 어텐션 점수를 구하고, 최근 recent_size개 토큰은 항상 유지되도록 무한대 점수를 부여합니다.

[요구사항]
- attn_kv [B, H, Q, K]에서 쿼리 축(dim=-2)을 따라 합산하여 키 토큰별 누적 중요도(scores) 계산
- 마지막 recent_size개 토큰 위치에 float('inf')를 할당하여 pruning 대상에서 제외

In [ ]:
# [코드]
def h2o_score(attn_kv, layer, budget):
    recent_size = budget // 2
    scores = ### 작성 필요 ###
    ### 작성 필요 ###
    return scores

## 문제 17. (d-3) SnapKV: Observation Window & Max Pooling 스코어링
[문제]
Q d-3) 마지막 w개 관측 쿼리의 어텐션 합산, 커널 범위 내 최댓값 풀링(Max Pooling), 관측 윈도우 영구 보존 로직을 완성합니다.

[요구사항]
- attn_kv [B, H, Q, K]에서 마지막 w개 쿼리 슬라이싱 후 쿼리 축(dim=2) 합산
- 각 위치 j 주변 [start, end) 슬라이스 내에서 마지막 차원 기준 최댓값 추출
- 마지막 w개 관측 윈도우 토큰에는 float('inf') 할당

In [ ]:
# [코드]
def snapkv_score(attn_kv, layer, budget, window=32, kernel=7):
    B, H, Q, K = attn_kv.shape
    w = min(window, Q)

    scores = ### 작성 필요 ###

    half = kernel // 2
    pooled_scores = scores.clone()
    for j in range(K):
        start = max(0, j - half)
        end = min(K, j + half + 1)
        pooled_scores[:, :, j] = ### 작성 필요 ###

    scores = pooled_scores
    ### 작성 필요 ###
    return scores

## 문제 18. (d-4) LLM Magnitude-based Pruning
[문제]
Q d-4) LLM 내 Linear 가중치 텐서별로 절댓값 하위 비율(sparsity) 임계값을 계산하고 불리언 마스크를 생성해 프루닝합니다.

[요구사항]
- 전체 원소 수에 sparsity를 곱해 반올림(round)하여 제거할 개수(num_zeros) 산출
- torch.abs(W)의 평탄화된 텐서에서 kthvalue를 이용해 임계값 계산 후 초과 원소만 True인 마스크 생성

In [ ]:
# [코드]
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            num_elements = W.numel()
            num_zeros = ### 작성 필요 ###
            importance = ### 작성 필요 ###
            threshold = ### 작성 필요 ###
            mask = ### 작성 필요 ###
            W.mul_(mask)

## 문제 19. (d-5) Wanda용 Activation L2 Norm 누적
[문제]
Q d-5) forward hook 내에서 전치된 입력 활성화 텐서 `x` (hidden_size, batch_size)의 채널별 L2 Norm 제곱합을 계산합니다.

[요구사항]
- x의 각 채널(dim=1 방향)에 대해 L2 Norm의 제곱(||x||_2^2)을 계산
- 반환 텐서 형상은 (hidden_size,) 1차원 벡터여야 함

In [ ]:
# [코드]
def compute_activation_norm_sq(x: torch.Tensor) -> torch.Tensor:
    # x.shape: (hidden_size, batch_size)
    x = x.type(torch.float32)
    activation_norm = ### 작성 필요 ###
    return activation_norm

## 문제 20. (d-6) Wanda Pruning (Weights × Activations)
[문제]
Q d-6) 가중치 절댓값과 활성화 통계(input_feat)의 곱으로 중요도를 계산하고, 행(row) 단위 임계값 기준으로 프루닝 마스크를 생성합니다.

[요구사항]
- 출력 행별 제거 원소 개수(col * sparsity 반올림) 계산
- |W| * input_feat[n] 으로 중요도 행렬 계산
- 각 행(dim=1)마다 kthvalue로 임계값 산출 후 (row, 1) 형태로 브로드캐스팅 비교하여 마스크 생성

In [ ]:
# [코드]
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            row, col = W.shape
            num_zeros_per_row = ### 작성 필요 ###
            importance = ### 작성 필요 ###
            threshold = ### 작성 필요 ###
            mask = ### 작성 필요 ###
            W.mul_(mask)

## 문제 21. (e-1) Vanilla Speculative Decoding: Draft Generation & Target Verification
[문제]
Q e-1) Draft 모델로 gamma개의 토큰을 생성하는 루프와 Target 모델로 병렬 검증하기 위한 입력 시퀀스 결합 및 argmax 예측 추출 코드를 완성합니다.

[요구사항]
- draft_tokens에서 greedy_step을 gamma번 반복 호출해 다음 토큰과 갱신된 cache를 리스트에 누적
- verify_draft에서 마지막 확정 토큰 x와 drafted 시퀀스를 마지막 차원(dim=-1)으로 연결(torch.cat)
- Target 모델 출력 logits[0]에서 마지막 차원 기준 argmax로 예측 토큰 ID 추출
- ※ 사전 정의 함수: greedy_step(model, x, cache) -> (next_token, new_cache) 가 제공됩니다.

In [ ]:
# [코드]
@torch.inference_mode()
def draft_tokens(model, x, cache, gamma):
    tokens = []
    for _ in range(gamma):
        x, cache = ### 작성 필요 ###
        tokens.append(x)
    return torch.cat(tokens, dim=-1), cache

@torch.inference_mode()
def verify_draft(target, x, drafted, cache):
    verify_ids = ### 작성 필요 ###
    result = target(verify_ids, past_key_values=cache, use_cache=True)
    preds = ### 작성 필요 ###
    return preds, result.past_key_values

## 문제 22. (e-2) EAGLE Tree Decoding: Attention Mask & Posterior Acceptance
[문제]
Q e-2) 트리 부모-자식 연결 관계로부터 어텐션 마스크를 구성하고, 검증 후 가장 길게 일치한 분기 인덱스와 채택 길이를 선택합니다.

[요구사항]
- tree_mask 구성 시 각 자식 노드(child) 행에 부모 노드(parent) 행의 마스크를 비트 OR(|=)로 병합 상속
- accept_lengths 중 최댓값을 가지는 최적 분기 인덱스(best_candidate) 및 해당 채택 길이(accept_length) 추출

In [ ]:
# [코드]
def build_tree_mask_and_positions(mask_index_list, total_tokens, device=None):
    tree_mask = torch.eye(total_tokens + 1, dtype=torch.bool, device=device)
    tree_mask[:, 0] = True
    for child, parent in enumerate(mask_index_list, start=1):
        ### 작성 필요 ###
    tree_position_ids = tree_mask.sum(dim=1).long() - 1
    tree_mask = tree_mask.float()[None, None]
    return tree_mask, tree_position_ids

def select_best_branch(accept_lengths):
    best_candidate = ### 작성 필요 ###
    accept_length = ### 작성 필요 ###
    return best_candidate, accept_length

## 문제 23. (e-3) DFlash: Parallel Block Drafting
[문제]
Q e-3) DFlash 드래프터 출력 은닉 상태에서 제안할 마지막 num_draft개 토큰 슬라이싱, 로짓 계산, 병렬 argmax 토큰 선택 코드를 완성합니다.

[요구사항]
- draft_hidden [B, seq_len, dim]에서 뒤쪽 num_draft개 위치만 슬라이싱
- model.compute_logits(draft_hidden, output_head) 호출로 어휘 로짓 계산
- torch.argmax를 마지막 차원(dim=-1)에 적용해 모든 드래프트 토큰을 동시 추출

In [ ]:
# [코드]
def parallel_block_draft(model, target_hidden, noise_embedding, draft_position_ids,
                         past_key_values_draft, output_head, verify_size):
    draft_hidden = model(
        target_hidden=target_hidden,
        noise_embedding=noise_embedding,
        position_ids=draft_position_ids,
        past_key_values=past_key_values_draft,
        use_cache=True,
    )
    num_draft = verify_size - 1
    draft_hidden = ### 작성 필요 ###
    draft_logits = ### 작성 필요 ###
    draft_tokens = ### 작성 필요 ###
    return draft_tokens